# 🐍 Python for Everybody — Course 5: Retrieving, Processing and Visualizing Data

> **Platform:** Coursera | **Instructor:** Dr. Charles Severance (University of Michigan)  
> **Level:** Intermediate | **Language:** Python 3

---

## 📋 About This Course

This capstone course brings together all previous skills to build a real-world application: a **web search engine** with a **PageRank algorithm** and an interactive **data visualization**. The project crawls websites, stores link relationships in a database, computes page importance scores iteratively, and renders results as an interactive graph in the browser.

---

## 🏗️ Project Architecture

```
spider.py   →   spider.sqlite   →   sprank.py   →   spjson.py   →   force.html
  (crawl)       (database)        (PageRank)       (export JSON)    (visualize)
```

## 📚 Table of Contents

| Script | Role | Concepts |
|--------|------|----------|
| `spider.py` | Web crawler | Sockets, BeautifulSoup, SQLite, link extraction |
| `sprank.py` | PageRank engine | Graph algorithms, iterative convergence |
| `spdump.py` | Database inspector | JOIN query, ranking display |
| `spreset.py` | Reset ranks | SQL UPDATE, restart algorithm |
| `spjson.py` | JSON exporter | D3.js data prep, force-directed graph |

---

## 🔄 How to Run the Full Pipeline

Run each script in order from your terminal inside the `pagerank/` folder:

```bash
# Step 1 — Crawl a website (run multiple times to gather more pages)
python3 spider.py

# Step 2 — Compute PageRank (run multiple times to converge)
python3 sprank.py

# Step 3 — Inspect current rankings
python3 spdump.py

# Step 4 — Export to JSON for visualization
python3 spjson.py

# Step 5 — Open in browser
open force.html

# Optional — Reset PageRank scores and start over
python3 spreset.py
```

---

## 📌 Script 1 — `spider.py` : Web Crawler

### Description
Crawls a website starting from a given URL. For each page it visits, it stores the HTML and extracts all outbound links — building a graph of the web in `spider.sqlite`. Runs are additive: restarting resumes from where it left off.

### Concepts Covered
- **`BeautifulSoup`** — extract all `<a>` tags and their `href` attributes
- **`urljoin` / `urlparse`** — resolve relative URLs to absolute
- **3-table schema** — `Pages`, `Links`, `Webs` in SQLite
- **`ORDER BY RANDOM()`** — pick a random uncrawled page each run
- **SSL bypass** — handle sites with certificate issues
- **Incremental crawling** — `INSERT OR IGNORE` prevents re-crawling

### Key Takeaway
> This is the core of how search engines work. Every page visit creates nodes and edges in a graph — the raw material for PageRank.

In [ ]:
# spider.py — Web crawler: visits pages and records links in SQLite
import sqlite3
import urllib.error
import ssl
from urllib.parse import urljoin, urlparse
from urllib.request import urlopen
from bs4 import BeautifulSoup

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

conn = sqlite3.connect('spider.sqlite')
cur = conn.cursor()

cur.execute('''CREATE TABLE IF NOT EXISTS Pages
    (id INTEGER PRIMARY KEY, url TEXT UNIQUE, html TEXT,
     error INTEGER, old_rank REAL, new_rank REAL)''')
cur.execute('''CREATE TABLE IF NOT EXISTS Links
    (from_id INTEGER, to_id INTEGER, UNIQUE(from_id, to_id))''')
cur.execute('''CREATE TABLE IF NOT EXISTS Webs (url TEXT UNIQUE)''')

cur.execute('SELECT id,url FROM Pages WHERE html is NULL and error is NULL ORDER BY RANDOM() LIMIT 1')
row = cur.fetchone()
if row is not None:
    print('Restarting existing crawl. Remove spider.sqlite to start fresh.')
else:
    starturl = input('Enter web url or enter: ')  # e.g. http://www.dr-chuck.com/
    if len(starturl) < 1: starturl = 'http://www.dr-chuck.com/'
    if starturl.endswith('/'): starturl = starturl[:-1]
    web = starturl
    if starturl.endswith('.htm') or starturl.endswith('.html'):
        pos = starturl.rfind('/')
        web = starturl[:pos]
    if len(web) > 1:
        cur.execute('INSERT OR IGNORE INTO Webs (url) VALUES (?)', (web,))
        cur.execute('INSERT OR IGNORE INTO Pages (url, html, new_rank) VALUES (?, NULL, 1.0)', (starturl,))
        conn.commit()

cur.execute('SELECT url FROM Webs')
webs = [str(row[0]) for row in cur]
print(webs)

many = 0
while True:
    if many < 1:
        sval = input('How many pages: ')
        if len(sval) < 1: break
        many = int(sval)
    many -= 1
    cur.execute('SELECT id,url FROM Pages WHERE html is NULL and error is NULL ORDER BY RANDOM() LIMIT 1')
    try:
        row = cur.fetchone()
        fromid = row[0]
        url = row[1]
    except:
        print('No unretrieved HTML pages found')
        break

    print(fromid, url, end=' ')
    cur.execute('DELETE from Links WHERE from_id=?', (fromid,))
    try:
        document = urlopen(url, context=ctx)
        html = document.read()
        if document.getcode() != 200:
            cur.execute('UPDATE Pages SET error=? WHERE url=?', (document.getcode(), url))
        if 'text/html' != document.info().get_content_type():
            cur.execute('DELETE FROM Pages WHERE url=?', (url,))
            conn.commit()
            continue
        print('(' + str(len(html)) + ')', end=' ')
        soup = BeautifulSoup(html, 'html.parser')
    except KeyboardInterrupt:
        break
    except:
        print('Unable to retrieve or parse page')
        cur.execute('UPDATE Pages SET error=-1 WHERE url=?', (url,))
        conn.commit()
        continue

    cur.execute('UPDATE Pages SET html=? WHERE url=?', (memoryview(html), url))
    conn.commit()

    tags = soup('a')
    count = 0
    for tag in tags:
        href = tag.get('href', None)
        if href is None: continue
        up = urlparse(href)
        if len(up.scheme) < 1: href = urljoin(url, href)
        ipos = href.find('#')
        if ipos > 1: href = href[:ipos]
        if href.endswith(('.png', '.jpg', '.gif')): continue
        if href.endswith('/'): href = href[:-1]
        if len(href) < 1: continue
        found = any(href.startswith(web) for web in webs)
        if not found: continue
        cur.execute('INSERT OR IGNORE INTO Pages (url, html, new_rank) VALUES (?, NULL, 1.0)', (href,))
        count += 1
        conn.commit()
        cur.execute('SELECT id FROM Pages WHERE url=? LIMIT 1', (href,))
        try:
            toid = cur.fetchone()[0]
        except:
            continue
        cur.execute('INSERT OR IGNORE INTO Links (from_id, to_id) VALUES (?, ?)', (fromid, toid))
    print(count)

cur.close()

['http://www.dr-chuck.com']
1 http://www.dr-chuck.com/ 12
2 http://www.dr-chuck.com/csev-blog/ 57
How many pages:


## 📌 Script 2 — `sprank.py` : PageRank Algorithm

### Description
Reads the link graph from `spider.sqlite` and runs the **PageRank algorithm** for a given number of iterations. Each page's rank is redistributed proportionally to pages it links to. Prints the average rank change per iteration — when this converges near zero, the algorithm has stabilized.

### Concepts Covered
- **PageRank algorithm** — Google's original formula for ranking web pages by link importance
- **Graph traversal** — working with nodes (pages) and edges (links)
- **Iterative convergence** — running until the average change drops below a threshold
- **Evaporation** — redistributing rank from pages with no outbound links
- **SQL UPDATE** — writing computed ranks back to the database

### Key Takeaway
> PageRank is the algorithm that made Google famous. The insight: a page is important if important pages link to it. Running more iterations refines accuracy.

In [ ]:
# sprank.py — PageRank: computes page importance from the link graph
import sqlite3

conn = sqlite3.connect('spider.sqlite')
cur = conn.cursor()

cur.execute('SELECT DISTINCT from_id FROM Links')
from_ids = [row[0] for row in cur]

to_ids = []
links = []
cur.execute('SELECT DISTINCT from_id, to_id FROM Links')
for row in cur:
    from_id, to_id = row[0], row[1]
    if from_id == to_id: continue
    if from_id not in from_ids: continue
    if to_id not in from_ids: continue
    links.append(row)
    if to_id not in to_ids: to_ids.append(to_id)

prev_ranks = {}
for node in from_ids:
    cur.execute('SELECT new_rank FROM Pages WHERE id = ?', (node,))
    prev_ranks[node] = cur.fetchone()[0]

sval = input('How many iterations: ')
many = int(sval) if len(sval) > 0 else 1

if len(prev_ranks) < 1:
    print('Nothing to page rank. Check data.')
    quit()

for i in range(many):
    next_ranks = {}
    total = 0.0
    for node, old_rank in prev_ranks.items():
        total += old_rank
        next_ranks[node] = 0.0

    for node, old_rank in prev_ranks.items():
        give_ids = [to_id for from_id, to_id in links if from_id == node and to_id in to_ids]
        if len(give_ids) < 1: continue
        amount = old_rank / len(give_ids)
        for id in give_ids:
            next_ranks[id] += amount

    newtot = sum(next_ranks.values())
    evap = (total - newtot) / len(next_ranks)
    for node in next_ranks:
        next_ranks[node] += evap

    totdiff = sum(abs(prev_ranks[n] - next_ranks[n]) for n in prev_ranks)
    avediff = totdiff / len(prev_ranks)
    print(i + 1, avediff)
    prev_ranks = next_ranks

print(list(next_ranks.items())[:5])
cur.execute('UPDATE Pages SET old_rank=new_rank')
for id, new_rank in next_ranks.items():
    cur.execute('UPDATE Pages SET new_rank=? WHERE id=?', (new_rank, id))
conn.commit()
cur.close()

1 0.546848992536
2 0.226714939664
3 0.065951618724
...
50 5.10410499467e-05
[(1, 12.79), (2, 28.93), (3, 6.80), (4, 13.46), (5, 0.65)]


## 📌 Script 3 — `spdump.py` : Database Inspector

### Description
Queries `spider.sqlite` and prints the top 50 pages sorted by number of inbound links, showing their old rank, new rank, id, and URL.

### Concepts Covered
- **JOIN query** — combining `Pages` and `Links` tables
- **`COUNT(from_id)`** — counting inbound links per page
- **`GROUP BY` / `ORDER BY DESC`** — aggregate and sort results
- **Reading query results** — iterating over cursor rows

In [ ]:
# spdump.py — Inspect database: show pages ranked by inbound links
import sqlite3

conn = sqlite3.connect('spider.sqlite')
cur = conn.cursor()

cur.execute('''SELECT COUNT(from_id) AS inbound, old_rank, new_rank, id, url
     FROM Pages JOIN Links ON Pages.id = Links.to_id
     WHERE html IS NOT NULL
     GROUP BY id ORDER BY inbound DESC''')

count = 0
for row in cur:
    if count < 50: print(row)
    count += 1
print(count, 'rows.')
cur.close()

(5, 1.0, 0.985, 3, 'http://www.dr-chuck.com/csev-blog')
(3, 1.0, 2.135, 4, 'http://www.dr-chuck.com/dr-chuck/resume/speaking.htm')
(1, 1.0, 0.659, 2, 'http://www.dr-chuck.com/csev-blog/')
(1, 1.0, 0.659, 5, 'http://www.dr-chuck.com/dr-chuck/resume/index.htm')
4 rows.


## 📌 Script 4 — `spreset.py` : Reset PageRank

### Description
Resets all page ranks to 1.0 in the database without re-crawling. Useful for restarting the PageRank calculation with a clean slate after adding more pages.

### Concepts Covered
- **`UPDATE` without WHERE** — modifying all rows in a table at once
- **`conn.commit()`** — persisting changes to disk

In [ ]:
# spreset.py — Reset all PageRank scores to 1.0
import sqlite3

conn = sqlite3.connect('spider.sqlite')
cur = conn.cursor()

cur.execute('UPDATE Pages SET new_rank=1.0, old_rank=0.0')
conn.commit()
cur.close()

print('All pages set to a rank of 1.0')

All pages set to a rank of 1.0


## 📌 Script 5 — `spjson.py` : JSON Exporter & Visualization

### Description
Exports the top N pages and their links from `spider.sqlite` into `spider.js` — a JSON file formatted for D3.js force-directed graph visualization. Open `force.html` in a browser to see the interactive graph.

### Concepts Covered
- **D3.js data format** — nodes and links as JSON
- **Rank normalization** — scaling ranks to a 0–19 range for visual weight
- **Writing JSON to file** — manual JSON construction for compatibility
- **Force-directed graph** — physics-based network visualization

### Key Takeaway
> This closes the full pipeline: raw web data → database → algorithm → interactive visualization. The output is a live graph where node size reflects PageRank importance.

In [ ]:
# spjson.py — Export top pages to JSON for D3.js force-directed visualization
import sqlite3

conn = sqlite3.connect('spider.sqlite')
cur = conn.cursor()

print('Creating JSON output on spider.js...')
howmany = int(input('How many nodes? '))

cur.execute('''SELECT COUNT(from_id) AS inbound, old_rank, new_rank, id, url
    FROM Pages JOIN Links ON Pages.id = Links.to_id
    WHERE html IS NOT NULL AND ERROR IS NULL
    GROUP BY id ORDER BY id, inbound''')

fhand = open('spider.js', 'w')
nodes = []
maxrank = minrank = None
for row in cur:
    nodes.append(row)
    rank = row[2]
    if maxrank is None or maxrank < rank: maxrank = rank
    if minrank is None or minrank > rank: minrank = rank
    if len(nodes) > howmany: break

if maxrank == minrank or maxrank is None:
    print('Error - please run sprank.py to compute page rank')
    quit()

fhand.write('spiderJson = {"nodes":[\n')
count = 0
nodemap = {}
ranks = {}
for row in nodes:
    if count > 0: fhand.write(',\n')
    rank = 19 * ((row[2] - minrank) / (maxrank - minrank))
    fhand.write('{"weight":' + str(row[0]) + ',"rank":' + str(rank) + ',')
    fhand.write(' "id":' + str(row[3]) + ', "url":"' + row[4] + '"}')
    nodemap[row[3]] = count
    ranks[row[3]] = rank
    count += 1
fhand.write('],\n')

cur.execute('SELECT DISTINCT from_id, to_id FROM Links')
fhand.write('"links":[\n')
count = 0
for row in cur:
    if row[0] not in nodemap or row[1] not in nodemap: continue
    if count > 0: fhand.write(',\n')
    fhand.write('{"source":' + str(nodemap[row[0]]) + ',"target":' + str(nodemap[row[1]]) + ',"value":3}')
    count += 1
fhand.write(']};')
fhand.close()
cur.close()

print('Open force.html in a browser to view the visualization')

Creating JSON output on spider.js...
Open force.html in a browser to view the visualization


---

## ✅ Course & Specialization Summary

| Concept | Script |
|---|---|
| Web crawling with BeautifulSoup | spider.py |
| Link graph stored in SQLite | spider.py |
| PageRank iterative algorithm | sprank.py |
| SQL JOIN aggregation queries | spdump.py |
| Bulk SQL UPDATE | spreset.py |
| JSON export for D3.js visualization | spjson.py |
| Force-directed graph in browser | force.html |

---

## 🎓 Python for Everybody — Full Specialization Complete!

| Course | Topic |
|---|---|
| Course 1 | Programming for Everybody — Python basics |
| Course 2 | Python Data Structures — strings, lists, dicts, tuples |
| Course 3 | Using Python to Access Web Data — regex, HTTP, APIs, XML, JSON |
| Course 4 | Using Databases with Python — SQLite, SQL, relational design |
| Course 5 | Retrieving, Processing and Visualizing Data — PageRank, D3.js |

> 🎓 **Certificate:** [Python for Everybody Specialization — Coursera](https://www.coursera.org/specializations/python)